# **Evaluation of Composed Image Retrieval with Multi Image Queries**

This notebook evaluates five strategies for multi image Composed Image Retrieval on the [PinPoint benchmark](https://github.com/pinterest/pinpoint-dataset) (CVPR 2026).

In multi image CIR, a query is formed from two reference images and a text instruction. The objective is to retrieve relevant images from a large corpus.
<br><br>
#### **Strategies**
| Method | Key idea |
|---|---|
| **Text Only (Baseline)** | Ignore reference images and database retrieval is dependent only on text instruction |
| **Mean Pooling** | Averages numerical vectors of both reference images equally and includes text instruction |
| **Max Pooling** | Compares both reference images and selects highest activation for each feature. Text instruction is combined with surfaced traits |
| **Weighted Pooling** | Custom attention mechanism that maps images and texts into same embedding space through OpenCLIP. Through comparison, higher weight is assigned to more relevant image with text instruction. |
| **LLM Translation** | Uses image captioning model (BLIP) to translate visual contents of references images into text. Appends user instruction with captions for text based retrieval. |
<br><br>

### **Setup**: Install dependencies and clone PinPoint Repo

In [ ]:
!git clone -q https://github.com/pinterest/pinpoint-dataset.git
%cd pinpoint-dataset/

/content/pinpoint-dataset


In [ ]:
!echo 'n' | bash setup.sh -y >/dev/null

In [ ]:
import os
import json
import torch
import random
import requests
import open_clip
import numpy as np
import pandas as pd

from tqdm import tqdm
from PIL import Image
from itertools import chain
from google.colab import userdata
from huggingface_hub import login
from transformers import BlipProcessor, BlipForConditionalGeneration

hf_token = userdata.get('hf')
login(token=hf_token)
IMG_DIR = '/content/drive/MyDrive/pinpoint_images/'

### **Data**: Filter for multi image queries

In [ ]:
df = pd.read_parquet('pinpoint_licensed.parquet')

multi = df[df['query_image_signature2'] != 'None']

print(len(df))
print(len(multi))

7635
1006


In [ ]:
# understanding dataset
sample = multi.iloc[0]
print(f"Query ID: {sample['query_id']}")
print(f"Instruction: {sample['instruction']}")
print(f"Query Image: {sample['query_image_signature']}")
print(f"Positive candidates: {sample['positive_candidates'][:3]}...")
print(f"Negative candidates: {sample['negative_candidates'][:3]}...")

def signature_to_url(signature):
    """Convert a Pinterest signature to a viewable image URL."""
    return f"https://i.pinimg.com/736x/{signature[:2]}/{signature[2:4]}/{signature[4:6]}/{signature}.jpg"

query = 'f8add0f0e359e6886b9762291aa621a4'
pos = '61ec3d06b7c08cf068f92f24e8721716'
neg = 'c9f146680bd90dee208037d77753f23a'

qurl = signature_to_url(query) # cozy sweater image
purl = signature_to_url(pos) # cozy bedroom layout image (positive)
nurl = signature_to_url(neg) # standard bedroom layout image (negative)

print(qurl)
print(purl)
print(nurl)

Query ID: query_00729
Instruction: bedroom, same aesthetic
Query Image: f8add0f0e359e6886b9762291aa621a4
Positive candidates: ['61ec3d06b7c08cf068f92f24e8721716' 'd46418c268b305f0fc3883a280fc5387'
 '8387de70709db6ec3eca3effaab97147']...
Negative candidates: ['c9f146680bd90dee208037d77753f23a' '7ef48e7228869f6bb451b5803510c6fc'
 '32330f0d6e7abeafc2e3299b24d680a8']...
https://i.pinimg.com/736x/f8/ad/d0/f8add0f0e359e6886b9762291aa621a4.jpg
https://i.pinimg.com/736x/61/ec/3d/61ec3d06b7c08cf068f92f24e8721716.jpg
https://i.pinimg.com/736x/c9/f1/46/c9f146680bd90dee208037d77753f23a.jpg


### **Images**: Fetch required signatures and additional background sample

Extracts all necessary image signatures required by the evaluation queries. This includes true targets and hard negatives. Merge required set with a randomized background sample from the full dataset. This introduces visual noise for unbiased retrieval to simulate a large scale search index with low resources.

In [ ]:
req = set(chain(
    multi['query_image_signature'],
    multi['query_image_signature2'],
    *multi['positive_candidates'],
    *multi['negative_candidates']
))

add = set(chain(
    df['query_image_signature'],
    df['query_image_signature2'],
    *df['positive_candidates'],
    *df['negative_candidates']
))

req.discard('None')
add.discard('None')

add = add - req # remove duplicates
add_subset = random.sample(list(add), k=2000)
sigs = list(req) + add_subset
print(len(sigs))

10296


In [ ]:
os.makedirs(IMG_DIR, exist_ok=True)

def download_images(sig):
  if not sig: return False

  path = f'{IMG_DIR}{sig}.jpg'
  if os.path.exists(path): return True

  try:
    res = requests.get(signature_to_url(sig), timeout=10)
    with open(path, 'wb') as f: f.write(res.content)
    return True
  except Exception:
    pass

for s in tqdm(sigs):
  download_images(s)

100%|██████████| 10296/10296 [37:34<00:00,  4.57it/s]


### **Embeddings**: Build searchable vector index with CLIP

This code uses OpenCLIP, a multi modal transformer architecture trained to project images and text into a shared latent space. This allows for cross modal retrieval, where images and text can be directly compared based on semantic similarity.

The image corpus is created from the earlier defined sigs database. There are helper functions to compute the feature embeddings for both images and text. The similarity between these embeddings is determined using cosine similarity. Before comparision, vectors are normalized. This similarity is determined by relative angular closeness between the query and the data in the latent space.

Note: Accessing public models via OpenCLIP does not require a Hugging Face token so intentionally running in an unauthenticated state.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai', device=device)
clip_model.eval()

def embed_image(sig):
    try:
        img_tensor = clip_preprocess(Image.open(f"{IMG_DIR}{sig}.jpg").convert('RGB')).unsqueeze(0).to(device)
        with torch.no_grad():
            emb = clip_model.encode_image(img_tensor)
            emb /= emb.norm(dim=-1, keepdim=True)
        return emb.squeeze().cpu().numpy()
    except:
        return None

def embed_text(text):
    tokens = open_clip.tokenize([text]).to(device)
    with torch.no_grad():
        emb = clip_model.encode_text(tokens)
        emb /= emb.norm(dim=-1, keepdim=True)
    return emb.squeeze().cpu().numpy()

corpus_embeddings = {}
for sig in tqdm(sigs):
    if sig and (emb := embed_image(sig)) is not None:
        corpus_embeddings[sig] = emb

corpus_sigs = list(corpus_embeddings.keys())
corpus_matrix = np.array([corpus_embeddings[s] for s in corpus_sigs])
print(corpus_matrix.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:476: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(
100%|██████████| 10296/10296 [04:15<00:00, 40.28it/s]


(10294, 512)


### **Strategies**: First four methods for multi image queries

**search_by_vector** represents the retrieval function. It computes the dot product between the query vector and the corpus_matrix. As all vectors are normalized, this represents Cosine Similarity. This acts as a simplified version of the OmniSage retrieval system described in the Pinterest Canvas paper

**mean_pooling** represents the baseline aggregation method. It computes the mean of two reference image embeddings to create a single representative vector. It then merges through linear combination with the text instructions (text_emb) weighted by alpha = 0.8.

**max_pooling** represents an alternative aggregation method using maximum. It extracts the strongest activation across the two input images for every dimension in the embedding space. The intent is to prevent the blurring effect of averaging, potentially preserving sharper visual signals that align better with the query

**weighted_pooling** represents an approximated dynamic attention method. It calculates the dot product (semantic alignment) of each image embedding with the text instruction. These scores determine the weights that forces the retrieval system to prioritize the reference image more semantically relevant to the user's prompt.

**text_only** represents the control baseline. It completely disregards the reference images and searches only from the instructional embedding. This is equivalent to the control arm mentioned in the paper. The performance delta, semantic relevance and product alignment, is measured from here by adding image based context.

In [ ]:
def search_by_vector(row, query: np.ndarray, k: int=50) -> list[str]:
  '''
  Apply cosine similarity across corpus
  Return top k image signatures ranked by similarity
  '''
  ignore_sigs = {row['query_image_signature'], row['query_image_signature2']}

  scores = corpus_matrix @ query
  top_indices = np.argsort(scores)[::-1]

  sigs = []
  for i in top_indices:
    s = corpus_sigs[i]
    if s in ignore_sigs:
      continue
    sigs.append(s)
    if len(sigs) == k:
      break

  return sigs

def mean_pooling(row, k: int=50) -> list[str]:
  '''
  Baseline from PinPoint paper
  Averages two reference image embeddings for search
  '''

  emb1 = embed_image(row['query_image_signature'])
  emb2 = embed_image(row['query_image_signature2'])
  text_emb = embed_text(row['instruction'])
  if emb1 is None or emb2 is None: return []

  res = (emb1 + emb2) / 2
  res = res / np.linalg.norm(res)

  alpha = 0.8
  query = (alpha * text_emb) + ((1 - alpha) * res)
  query = query / np.linalg.norm(query)

  return search_by_vector(row, query, k)

def max_pooling(row, k: int=50) -> list[str]:
  '''
  Element wise maximum of two reference image embeddings for search
  Dominant visual feature is more likely to be relevant signal than average?
  '''

  emb1 = embed_image(row['query_image_signature'])
  emb2 = embed_image(row['query_image_signature2'])
  text_emb = embed_text(row['instruction'])
  if emb1 is None or emb2 is None: return []

  res = np.maximum(emb1, emb2)
  res = res / np.linalg.norm(res)

  alpha = 0.8
  query = (alpha * text_emb) + ((1 - alpha) * res)
  query = query / np.linalg.norm(query)

  return search_by_vector(row, query, k)

def weighted_pooling(row, k: int = 50) -> list[str]:
  '''
  Weighs each image embedding by alignment with text instruction in CLIP embedding space
  '''

  emb1 = embed_image(row['query_image_signature'])
  emb2 = embed_image(row['query_image_signature2'])
  text_emb = embed_text(row['instruction'])
  if emb1 is None or emb2 is None: return []

  sim1 = float(np.dot(emb1, text_emb))
  sim2 = float(np.dot(emb2, text_emb))

  total = abs(sim1) + abs(sim2)
  w1, w2 = abs(sim1) / total, abs(sim2) / total

  res = w1*emb1 + w2*emb2
  res = res / np.linalg.norm(res)

  alpha = 0.8
  query = (alpha * text_emb) + ((1 - alpha) * res)
  query = query / np.linalg.norm(query)

  return search_by_vector(row, query, k)

def text_only(row, k: int=50) -> list[str]:
  '''
  Baseline from PinPoint paper
  Ignores both reference images and retrieves using only text instruction
  '''
  text_emb = embed_text(row['instruction'])
  return search_by_vector(row, text_emb, k)

### **Strategies**: 5th method using BLIP captioning for visual translation

Instead of relying on raw image embeddings, this pipeline translates the visual content from the query images into natural language descriptions. These generated captions are then combined with the text instruction, giving the prompt more context. This allows the CLIP text encoder to reason with detailed, structured linguistic instructions over image based feature matching.

In [ ]:
blip_processor = BlipProcessor.from_pretrained('Salesforce/blip-image-captioning-large')
blip_model = BlipForConditionalGeneration.from_pretrained('Salesforce/blip-image-captioning-large').to(device)
blip_model.eval()

cache: dict[str, str] = {}

def caption_image(sig: str) -> str:
  '''
  Create a NL caption for an image using Blip
  Results are cached so each image is captioned once
  '''

  if sig in cache:
    return cache[sig]

  path = f'{IMG_DIR}{sig}.jpg'
  if not os.path.exists(path): return ''

  try:
        raw_image = Image.open(path).convert('RGB')
        inputs    = blip_processor(raw_image, return_tensors="pt").to(device)
        with torch.no_grad():
            output = blip_model.generate(**inputs, max_new_tokens=30)

        caption = blip_processor.decode(output[0], skip_special_tokens=True).strip().lower()

        for prefix in ('a picture of', 'an image of'):
            if caption.startswith(prefix):
                caption = caption[len(prefix):].strip()

        cache[sig] = caption
        return caption

  except Exception as e:
        print(f'Caption error for {sig[:6]}: {e}')
        return ''

def llm(row, k: int=50) -> list[str]:
  '''
  Use Blip to caption both referenced image embeddings
  Include captions to text instruction before encoding with Clip
  Clip understands detailed descriptions over image encoder
  Let text encoder handle language converted from visual content instead
  '''
  caption1 = caption_image(row['query_image_signature'])
  caption2 = caption_image(row['query_image_signature2'])

  visual_context = ' mixed with '.join(c for c in [caption1, caption2] if c)
  if visual_context: prompt = f"{row['instruction']}. Must match this visual style: {visual_context}"
  else: prompt = row['instruction']

  return search_by_vector(row, embed_text(prompt), k)

preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.60k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/527 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.88G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/616 [00:00<?, ?it/s]

### **Retrivals**: Apply all methods on the filtered multi image queries

In [ ]:
def format(id: str) -> str:
  '''
  Remove 'query_' prefix from IDs to match PinPoint's evaluation format
  '''
  return id.replace('query_', '')

results: dict[str, dict] = {
    'mean': {},
    'max': {},
    'wp': {},
    'text': {},
    'llm': {},
}

for _, row in tqdm(multi.iterrows(), total=len(multi)):
    qid = format(row['query_id'])
    results['mean'][qid] = {"retrieved_items": mean_pooling(row)}
    results['max'][qid]  = {"retrieved_items": max_pooling(row)}
    results['wp'][qid]   = {"retrieved_items": weighted_pooling(row)}
    results['text'][qid] = {"retrieved_items": text_only(row)}
    results['llm'][qid]  = {"retrieved_items": llm(row)}

for method_name, result_dict in results.items():
    with open(f"results_{method_name}.json", "w") as f:
        json.dump(result_dict, f)

100%|██████████| 1006/1006 [06:14<00:00,  2.68it/s]


### **Evaluate**: Score each method with PinPoint Evaluator

In [ ]:
multi.to_parquet('pinpoint_multi.parquet')

for method in results:
  !python evaluate.py --results results_{method}.json --ground_truth pinpoint_multi.parquet --output metrics_{method}.csv

Loading ground truth...
Loaded 1006 queries
Found 1 result file(s)

Processing results_mean...
Evaluating results_mean...
  Matched 1006/1006 queries

✅ Saved metrics to metrics_mean.csv
Loading ground truth...
Loaded 1006 queries
Found 1 result file(s)

Processing results_max...
Evaluating results_max...
  Matched 1006/1006 queries

✅ Saved metrics to metrics_max.csv
Loading ground truth...
Loaded 1006 queries
Found 1 result file(s)

Processing results_wp...
Evaluating results_wp...
  Matched 1006/1006 queries

✅ Saved metrics to metrics_wp.csv
Loading ground truth...
Loaded 1006 queries
Found 1 result file(s)

Processing results_text...
Evaluating results_text...
  Matched 1006/1006 queries

✅ Saved metrics to metrics_text.csv
Loading ground truth...
Loaded 1006 queries
Found 1 result file(s)

Processing results_llm...
Evaluating results_llm...
  Matched 1006/1006 queries

✅ Saved metrics to metrics_llm.csv


## **Results**

###**Metrics**
- **Mean Average Precision:** System's ability to prioritize the most relevant results at the top of the list
- **Negative Recall:** System's ability to exclude irrelevant images from the retrieved set
- **Precision Density:** Measures how tightly the relevant results are packed together at the top of the list rather than being scattered
- **Robustness (Std):** Measures how reliably the system performs across a wide variety of different queries

###**Findings**
- **Multimodal Conditioning:** All multimodal methods yielded a higher Mean Average Precision (MAP) than the text only baseline (0.0330 MAP). This result is consistent with the paper's thesis that multimodal conditioning helps ensure retrieval remains aligned with the user's specific visual intent.

- **Feature Combination:** The max (0.0420 MAP) and weighted_pooling (0.0419 MAP) methods achieved the highest retrieval accuracy. While the performance gap between these and mean pooling (0.0416 MAP) is narrow, it suggests that techniques which isolate dominant features or prioritize semantic alignment may offer greater precision. Although the paper supports mean aggregation, the discrepancy in my experimental findings may be due to the evaluation on a smaller ground truth set.

- **Stability vs. Accuracy Tradeoffs:** While the llm captioning approach resulted in lower peak accuracy (0.0376 MAP), it achieved the highest consistency, evidenced by the lowest Robustness Std (0.0220). This performance deficit can be attributed to the inherent limitations of the BLIP model used for feature translation, where converting visual input into language leads to a loss of nuance otherwise preserved in raw embeddings. By abstracting images into text, the system sacrifices raw precision for a more predictable retrieval profile. This reflects a key tradeoff noted in the Pinterest Canvas production analysis, where minimizing performance variance is often as vital as achieving peak accuracy in production.

In [ ]:
rows = []

for method in results:
  m = pd.read_csv(f'metrics_{method}.csv')
  rows.append({
      'Method': method,
      'Mean Average Precision': m['mAP@10'].values[0],
      'Negative Recall': m['NegRecall@10'].values[0],
      'Precision Density': m['precision@10'].values[0],
      'Robustness Std': m['ling_sens_std'].values[0],
  })

res = pd.DataFrame(rows).sort_values('Mean Average Precision', ascending=False)
print(res.to_string(index=False))

Method  Mean Average Precision  Negative Recall  Precision Density  Robustness Std
   max                  0.0420           0.1336             0.0596          0.0293
    wp                  0.0419           0.1361             0.0609          0.0289
  mean                  0.0416           0.1342             0.0599          0.0275
   llm                  0.0376           0.1531             0.0541          0.0220
  text                  0.0330           0.1075             0.0483          0.0285
